# NumPy Basics and Validation with IMF WEO Data

This notebook teaches NumPy using the same IMF WEO Excel dataset as the pandas basics lab.

The goal is to understand what the array looks like, how `axis` works, and how NumPy helps validate financial/economic data quickly.

## 0. Setup

Run this cell in a new environment such as Google Colab. If packages already exist, it completes quickly.

In [ ]:
%pip install pandas numpy openpyxl -q

from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

## 1. Load the WEO Dataset

The normal path is `data/WEOApr2026all.xlsx`. In Colab, upload `WEOApr2026all.xlsx` into `/content/`.

If the file is not available, the notebook creates a small fallback dataset so the NumPy concepts still run.

In [ ]:
course_path = Path("data/WEOApr2026all.xlsx")
colab_path = Path("/content/WEOApr2026all.xlsx")
excel_path = course_path if course_path.exists() else colab_path if colab_path.exists() else None

if excel_path:
    raw_df = pd.read_excel(excel_path, sheet_name="Countries")
    print(f"Loaded WEO file: {excel_path}")
else:
    print("WEO file not found. Using small fallback data.")
    raw_df = pd.DataFrame({
        "COUNTRY.ID": ["ZWE", "ZAF", "BWA", "LSO"],
        "COUNTRY": ["Zimbabwe", "South Africa", "Botswana", "Lesotho, Kingdom of"],
        "INDICATOR.ID": ["PCPIPCH", "PCPIPCH", "PCPIPCH", "PCPIPCH"],
        "INDICATOR": ["All Items, Consumer price index (CPI), Period average, percent change"] * 4,
        "UNIT": ["Percent"] * 4,
        2020: [557.2, 3.2, 1.9, 5.0],
        2021: [98.5, 4.6, 6.7, 6.0],
        2022: [193.4, 6.9, 12.2, np.nan],
        2023: [667.4, 6.0, 5.1, 6.4],
        2024: [57.5, 4.4, 2.8, 6.1],
        2025: [21.0, 4.0, 3.5, 5.8],
        2026: [10.0, 4.5, 4.0, 5.5]
    })

raw_df.head()

## 2. Prepare a Small Analytical DataFrame

NumPy works best when the input values are numeric. We use pandas briefly to clean column names, select inflation rows, and convert year columns.

In [ ]:
df = raw_df.copy()
df.columns = (
    df.columns.astype(str)
    .str.strip()
    .str.lower()
    .str.replace(".", "_", regex=False)
    .str.replace(" ", "_", regex=False)
)
df = df.rename(columns={"country_id": "country_code", "indicator_id": "indicator_code"})

year_cols = [str(year) for year in range(2020, 2027) if str(year) in df.columns]
df[year_cols] = df[year_cols].apply(pd.to_numeric, errors="coerce")

countries = ["Zimbabwe", "South Africa", "Botswana", "Lesotho", "Lesotho, Kingdom of", "Namibia", "Zambia"]
weo_df = df[df["country"].isin(countries) & (df["indicator_code"] == "PCPIPCH")].copy()

weo_df[["country", "indicator_code"] + year_cols]

## 3. What Does `.to_numpy()` Mean?

`.to_numpy()` converts selected pandas columns into a NumPy array.

In simple terms:

```text
Pandas DataFrame -> NumPy array
```

If you select several year columns, NumPy sees them as a table of numbers.

In [ ]:
data_array = weo_df[year_cols].to_numpy(dtype=float)

print(data_array)
print("Array type:", type(data_array))

## 4. Check the Shape of the Array

`shape` tells you the structure of the array.

```text
(rows, columns)
```

Rows are countries. Columns are years.

In [ ]:
print("Shape:", data_array.shape)
print("Rows/countries:", data_array.shape[0])
print("Columns/years:", data_array.shape[1])

## 5. Overall Mean, Mean Per Year, and Mean Per Country

`np.nanmean()` ignores missing values.

- No axis: calculate one overall mean using every number.
- `axis=0`: calculate down each column, so one average per year.
- `axis=1`: calculate across each row, so one average per country.

In [ ]:
overall_mean = np.nanmean(data_array)
mean_per_year = np.nanmean(data_array, axis=0)
mean_per_country = np.nanmean(data_array, axis=1)

print("Overall mean:", round(overall_mean, 2))
print("Mean per year:", np.round(mean_per_year, 2))
print("Mean per country:", np.round(mean_per_country, 2))

## 6. Put NumPy Results Back Beside Labels

NumPy arrays do not keep pandas column names and country names. Use pandas again when you want labelled output.

In [ ]:
year_summary = pd.DataFrame({
    "year": year_cols,
    "mean_inflation": np.round(mean_per_year, 2)
})

country_summary = pd.DataFrame({
    "country": weo_df["country"].to_numpy(),
    "mean_inflation": np.round(mean_per_country, 2)
})

display(year_summary)
display(country_summary)

## 7. Understand Slicing: `data_array[:, 1:]` and `data_array[:, :-1]`

The square brackets mean:

```text
data_array[rows, columns]
```

- `:` means all rows.
- `1:` means columns from position 1 to the end.
- `:-1` means all columns except the last column.

This lets us compare each year with the previous year.

In [ ]:
new_year_values = data_array[:, 1:]
previous_year_values = data_array[:, :-1]

print("New year values shape:", new_year_values.shape)
print("Previous year values shape:", previous_year_values.shape)

print("New year values first row:", new_year_values[0])
print("Previous year values first row:", previous_year_values[0])

## 8. Year-on-Year Change

This expression:

```python
data_array[:, 1:] - data_array[:, :-1]
```

means:

```text
2021 - 2020
2022 - 2021
2023 - 2022
...
```

In [ ]:
change_array = data_array[:, 1:] - data_array[:, :-1]
change_year_labels = [f"{year_cols[i]}_to_{year_cols[i + 1]}" for i in range(len(year_cols) - 1)]

change_df = pd.DataFrame(change_array, columns=change_year_labels)
change_df.insert(0, "country", weo_df["country"].to_numpy())
change_df

## 9. Percentage Change

Formula:

```text
(new year - old year) / old year * 100
```

`np.errstate()` keeps divide-by-zero warnings from interrupting the lesson. We then replace infinite values with `NaN`.

In [ ]:
with np.errstate(divide="ignore", invalid="ignore"):
    pct_change_array = (change_array / previous_year_values) * 100

pct_change_array = np.where(np.isfinite(pct_change_array), pct_change_array, np.nan)

pct_change_df = pd.DataFrame(np.round(pct_change_array, 2), columns=change_year_labels)
pct_change_df.insert(0, "country", weo_df["country"].to_numpy())
pct_change_df

## 10. Missing Values and Basic Validation

NumPy is useful for quick validation checks across many numeric columns.

In [ ]:
missing_count = np.isnan(data_array).sum()
missing_by_year = np.isnan(data_array).sum(axis=0)
missing_by_country = np.isnan(data_array).sum(axis=1)

print("Total missing numeric values:", missing_count)
display(pd.DataFrame({"year": year_cols, "missing_values": missing_by_year}))
display(pd.DataFrame({"country": weo_df["country"].to_numpy(), "missing_values": missing_by_country}))

## 11. Min, Max, Standard Deviation, and Range

These checks help describe the spread of values.

In [ ]:
summary_by_country = pd.DataFrame({
    "country": weo_df["country"].to_numpy(),
    "min_value": np.nanmin(data_array, axis=1),
    "max_value": np.nanmax(data_array, axis=1),
    "range": np.nanmax(data_array, axis=1) - np.nanmin(data_array, axis=1),
    "std_dev": np.nanstd(data_array, axis=1)
})

summary_by_country.sort_values("range", ascending=False)

## 12. Z-Scores and Outlier Flags

A z-score shows how far a value is from the mean in standard deviation units.

A common rule is: if `abs(z_score) > 2`, review the value as a possible outlier. With small classroom data this is a simple demonstration, not a final audit rule.

In [ ]:
mean_2026 = np.nanmean(data_array[:, -1])
std_2026 = np.nanstd(data_array[:, -1])
z_scores_2026 = (data_array[:, -1] - mean_2026) / std_2026

outlier_review = pd.DataFrame({
    "country": weo_df["country"].to_numpy(),
    "value_2026": data_array[:, -1],
    "z_score_2026": np.round(z_scores_2026, 2),
    "review_flag": np.abs(z_scores_2026) > 2
})

outlier_review.sort_values("z_score_2026", ascending=False)

## 13. Boolean Masks

A Boolean mask is an array of `True` and `False` values. It lets you filter values based on a condition.

In [ ]:
high_2026_mask = data_array[:, -1] > 5

pd.DataFrame({
    "country": weo_df["country"].to_numpy(),
    "inflation_2026": data_array[:, -1],
    "above_5_percent": high_2026_mask
})

## 14. Export NumPy Validation Results

The final outputs are DataFrames because they keep country and year labels. NumPy performs the fast calculations; pandas stores the labelled result.

In [ ]:
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

country_summary_path = output_dir / "weo_numpy_country_summary.csv"
change_path = output_dir / "weo_numpy_year_on_year_change.csv"
outlier_path = output_dir / "weo_numpy_outlier_review.csv"

summary_by_country.to_csv(country_summary_path, index=False)
change_df.to_csv(change_path, index=False)
outlier_review.to_csv(outlier_path, index=False)

country_summary_path, change_path, outlier_path

## Key Ideas to Remember

| Concept | Meaning |
|---|---|
| `.to_numpy()` | Converts selected pandas columns into a NumPy array |
| `shape` | Shows `(rows, columns)` |
| `axis=0` | Calculate down each column, usually one result per year |
| `axis=1` | Calculate across each row, usually one result per country |
| `:` | Take everything in that direction |
| `1:` | Start from position 1 to the end |
| `:-1` | Take everything except the last item |
| `np.nanmean()` | Mean that ignores missing values |
| `np.isnan()` | Finds missing numeric values |
| Boolean mask | Filters values using `True`/`False` conditions |

NumPy is useful because it validates calculations quickly across many numeric columns, especially averages, changes, percentage changes, missing values, and outliers.